# The arc in the Abell 3128 sightline
**Biswajit Jana · 365 Earthian Days, Day 5**

Measure the angular location of the published DES J0330-5228 arc in Euclid and DECam. Compare two redshift geometries with the same circular, one-plane mass proxy.

In [ ]:
import importlib.util, json, subprocess, sys
from pathlib import Path
from urllib.request import urlopen
dependencies = ("astropy", "numpy", "scipy", "pandas", "matplotlib")
missing = [item for item in dependencies if importlib.util.find_spec(item) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
root = Path.cwd()
source_files = ("archive.py", "get_arc_fits.py", "arc_geometry.py", "lens_budget.py",
                "inputs/arc_image_manifest.json")
if not all((root / name).is_file() for name in source_files):
    root = Path("abell3128_replay").resolve()
    base = ("https://raw.githubusercontent.com/Biswajit1999/daily-astro-notebooks/"
            "research/euclid-dr1-foundation-day01-20261114/euclid-dr1-foundation/euclid-dark-universe/")
    for name in source_files:
        path = root / name
        path.parent.mkdir(parents=True, exist_ok=True)
        with urlopen(base + name, timeout=30) as response:
            path.write_bytes(response.read())
sys.path.insert(0, str(root))
import get_arc_fits
import arc_geometry

## Retrieve the four FITS cutouts
The manifest selects the exact mosaic hashes and byte ranges. The total transferred mosaic pixel rows are about 151 MB before any retries. Set ABELL3128_FITS_DIR to reuse the output of the retrieval script.

In [ ]:
from os import environ
cached = environ.get("ABELL3128_FITS_DIR")
inputs = Path(cached) if cached else root / "arc_inputs"
if not cached:
    get_arc_fits.retrieve(inputs, root / "inputs/arc_image_manifest.json")
print("Cutout directory:", inputs)

## Measure the ridge across ten sectors
Extract radial intensity profiles on the sky through the FITS WCS. Subtract a control profile and measure the strongest local maximum within the declared radial window.

In [ ]:
from IPython.display import Image, display
results = root / "arc_replay_results"
results.mkdir(exist_ok=True)
summary = arc_geometry.analyse(inputs, results)
display(Image(filename=str(results / "Abell3128_Observed_Arc.png")))

## Compare geometry at the same angle
Nord et al. report the arc source’s spectroscopic redshift. At foreground and background lens redshifts, the same measured angle corresponds to different physical radii and lensing masses.

In [ ]:
import pandas as pd
geometry = pd.DataFrame({
    "Background lens": summary["cluster_redshift_geometry"],
    "Foreground counterfactual": summary["foreground_counterfactual"],
}).T
display(geometry)

The foreground counterfactual isolates the role of lens distance. An actual two-cluster model would require additional lensing constraints. Sources: [Nord et al.](https://doi.org/10.3847/0004-637X/827/1/51), [McCleary et al.](https://arxiv.org/abs/1503.02823), and [Euclid DR1-Foundation](https://irsa.ipac.caltech.edu/data/Euclid/docs/overview_dr1.html).